[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Classification


## What you will be able to do

Train LightGBM and XGBoost to answer frost or no frost, and which band of cold, read the
probabilities they give, and judge them with accuracy, log loss and AUC on stations they never saw.

## The idea

### The problem

The question a gardener asks of a forecast is whether it will freeze tonight, and the question a road
service asks is how hard: a mild night, a frosty one, or one below −10 degrees that needs salt on
every road. A regression model gives a temperature, and −0.2 and +0.3 degrees are half a degree apart and
on opposite sides of the answer that matters. What either user can act on is a chance. At 0.9 the
plants get covered, and at 0.55 someone decides how much a lost crop is worth.

A model fitted to labels of 0 and 1 with squared error gives numbers that can fall below 0 or above
1, as the last of the Common errors shows, and a classifier is built to give probabilities instead.

### A formal definition

> A **classifier** predicts which of several classes a row belongs to. A gradient-boosted
> classifier's trees add up to a **raw score**, a single one when there are two classes and one for
> each class when there are more, and the raw scores become **probabilities** through the sigmoid
> function for two classes or the softmax for more. Training lowers the
> **log loss**, the average over the rows of minus the log of the probability the model gave the
> class that was true.

### Why it works that way

Boosting needs something for each new tree to fit, and for log loss the gradient for a row is its
predicted probability minus its label. Each tree is fitted to how far the probabilities are from the
labels, which is the residual of the **Why Gradient Boosting** notebook with a probability in place of
a temperature. The trees add to a raw score that can be any number, and the sigmoid turns any raw
score into a probability between 0 and 1. With more than two classes every round grows one tree for
each class, and the softmax turns the classes' raw scores into probabilities that add up to 1.

`predict` turns the probabilities into a class, at 0.5 for two classes and the most probable class for
more, and `predict_proba` keeps the probabilities. Accuracy counts the labels a classifier got right.
Log loss scores the probabilities, and punishes a confident mistake far more than a hesitant one.
AUC, the area under the ROC curve, asks how often a frost day gets a higher probability than a day
without frost, whatever threshold is chosen later.

### Where this shows up

The **Imbalanced Classes** notebook asks about hard frost, which almost never happens, and the
**A Frost Forecast** notebook builds the frost classifier from start to finish. The scikit-learn
guide's **Classification Metrics** notebook covers the scores in more depth.

### What this notebook covers

- frost and bands of cold, made from the lowest temperature
- `LGBMClassifier` and `XGBClassifier`, with `predict` and `predict_proba`
- accuracy, log loss and AUC on the validation stations
- the raw scores behind the probabilities, and the native interfaces
- bands of cold, with a tree for each class every round
- whether the probabilities match how often frost happens
- four mistakes made when training or scoring a classifier

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd
import lightgbm as lgb

days = pd.DataFrame({"cloud_cover": [i / 199 for i in range(200)]})
frost = (days["cloud_cover"] < 0.4).astype(int)
model = lgb.LGBMClassifier(n_estimators=50, verbose=-1).fit(days, frost)

nights = pd.DataFrame({"cloud_cover": [0.1, 0.9]})
print(model.predict_proba(nights).round(3))
print(model.predict(nights))
```

```
[[0.004 0.996]
 [0.997 0.003]]
[1 0]
```

In this made-up example every day with less than 0.4 of the sky clouded has frost. `predict_proba`
gives a row for each night and a column for each class, no frost and then frost, and each row adds up
to 1. The clear night gets a probability of frost of 0.996, and `predict` turns that into the label 1.

## Setup

Two libraries for the table, the two boosting libraries, and three scores and a confusion matrix from
scikit-learn.

- `numpy` and `pandas` build the table and the labels
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `accuracy_score`, `log_loss` and `roc_auc_score` score a classifier's labels and probabilities, and
  `confusion_matrix` counts its right and wrong answers by class

`station_days` makes 20,000 rows of daily weather at 400 weather stations in Norway, the table of the
**Why Gradient Boosting** notebook. The stations and their days are made up, and a generator started
from a fixed seed draws them, so the table is the same on every run. `FEATURES` names the eight
columns a model is given, and the cell splits the stations into 300 for training, 50 for validation and
50 for testing, keeping all of a station's days on one side. This notebook trains on the training
stations and scores every model on the validation stations.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, roc_auc_score

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### Frost from the lowest temperature

A day has frost when its lowest temperature is below 0. The labels are 1 for frost and 0 for none.

In [2]:
frost_train = (train["tmin"] < 0).astype(int)
frost_valid = (valid["tmin"] < 0).astype(int)
print("share of days with frost:", round(frost_train.mean(), 3), "training,", round(frost_valid.mean(), 3), "validation")

share of days with frost: 0.548 training, 0.528 validation


Frost falls on 54.8% of the training days and 52.8% of the validation days, so the two classes are
close to even.

### The classifiers and what they predict

`LGBMClassifier` and `XGBClassifier` take the same table as the regressors, with the labels as the
target.

In [3]:
lightgbm_frost = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_frost.fit(train[FEATURES], frost_train)
xgboost_frost = xgb.XGBClassifier(n_estimators=300, learning_rate=0.1, random_state=0)
xgboost_frost.fit(train[FEATURES], frost_train)

for name, model in (("LightGBM", lightgbm_frost), ("XGBoost", xgboost_frost)):
    probability = model.predict_proba(valid[FEATURES])[:, 1]
    label = model.predict(valid[FEATURES])
    print(name, "classes", model.classes_, "| probability of frost", probability[:4].round(3), "| label", label[:4],
          "| label is probability above 0.5:", bool((label == (probability > 0.5)).all()))

LightGBM classes [0 1] | probability of frost [0.323 1.    0.164 0.983] | label [0 1 0 1] | label is probability above 0.5: True
XGBoost classes [0 1] | probability of frost [0.605 0.999 0.143 0.904] | label [1 1 0 1] | label is probability above 0.5: True


Both models learned the classes 0 and 1 from the labels, and on every validation day `predict` is the
probability of frost turned into a label at 0.5. The first day gets 0.323 from LightGBM and 0.605
from XGBoost, so the two models give it opposite labels.

### Accuracy, log loss and AUC

Each score is computed on the validation stations, accuracy from the labels and log loss and AUC from
the probabilities of frost.

In [4]:
for name, model in (("LightGBM", lightgbm_frost), ("XGBoost", xgboost_frost)):
    probability = model.predict_proba(valid[FEATURES])[:, 1]
    label = model.predict(valid[FEATURES])
    print(f"{name:8} accuracy {accuracy_score(frost_valid, label):.3f} | log loss "
          f"{log_loss(frost_valid, probability):.3f} | AUC {roc_auc_score(frost_valid, probability):.3f}")
    print("  confusion matrix, rows true, columns predicted:", confusion_matrix(frost_valid, label).tolist())

LightGBM accuracy 0.945 | log loss 0.146 | AUC 0.989
  confusion matrix, rows true, columns predicted: [[1124, 57], [80, 1239]]
XGBoost  accuracy 0.946 | log loss 0.143 | AUC 0.989
  confusion matrix, rows true, columns predicted: [[1124, 57], [77, 1242]]


The two models score almost the same, with accuracy 0.945 and 0.946, log loss 0.146 and 0.143, and
an AUC of 0.989 for both, which one split of the stations cannot rank. In LightGBM's confusion matrix
the first row holds the 1,181 validation days without frost, 1,124 of them called correctly and 57
called frost, and the second row the 1,319 frost days, 80 of them missed.

### The raw scores behind the probabilities

LightGBM's `predict` with `raw_score=True` and XGBoost's with `output_margin=True` return each day's raw
score before the sigmoid, the starting score the model began from plus the sum of its trees. Put
through the sigmoid, 1 / (1 + e^−raw score), the raw scores give the probabilities again.

In [5]:
def sigmoid(raw_score):
    return 1 / (1 + np.exp(-raw_score))

lightgbm_raw = lightgbm_frost.predict(valid[FEATURES], raw_score=True)
xgboost_raw = xgboost_frost.predict(valid[FEATURES], output_margin=True)
print("LightGBM raw scores", lightgbm_raw[:3].round(3), "| sigmoid gives the probabilities:",
      np.allclose(sigmoid(lightgbm_raw), lightgbm_frost.predict_proba(valid[FEATURES])[:, 1]))
print("XGBoost raw scores ", xgboost_raw[:3].round(3), "| sigmoid gives the probabilities:",
      np.allclose(sigmoid(xgboost_raw), xgboost_frost.predict_proba(valid[FEATURES])[:, 1]))

LightGBM raw scores [-0.739  9.19  -1.626] | sigmoid gives the probabilities: True
XGBoost raw scores  [ 0.427  6.814 -1.787] | sigmoid gives the probabilities: True


A raw score of 9.19 is a probability of almost 1, and a raw score of −1.626 a probability of 0.164. In
both libraries the sigmoid of every raw score gives `predict_proba`'s probability, so a probability of
0.5 is a raw score of 0.

### The native interfaces

In the native interfaces the objective names the classifier, `"binary"` for LightGBM and
`"binary:logistic"` for XGBoost, and a booster's `predict` returns the probability of the class
labeled 1.

In [6]:
lightgbm_booster = lgb.train({"objective": "binary", "learning_rate": 0.1, "seed": 0, "verbose": -1},
                             lgb.Dataset(train[FEATURES], frost_train), num_boost_round=300)
xgboost_booster = xgb.train({"objective": "binary:logistic", "learning_rate": 0.1, "seed": 0},
                            xgb.DMatrix(train[FEATURES], frost_train, enable_categorical=True), num_boost_round=300)

lightgbm_native = lightgbm_booster.predict(valid[FEATURES])
xgboost_native = xgboost_booster.predict(xgb.DMatrix(valid[FEATURES], enable_categorical=True))
print("LightGBM", lightgbm_native[:3].round(3), "same as the classifier:",
      np.allclose(lightgbm_native, lightgbm_frost.predict_proba(valid[FEATURES])[:, 1]))
print("XGBoost ", xgboost_native[:3].round(3), "same as the classifier:",
      np.allclose(xgboost_native, xgboost_frost.predict_proba(valid[FEATURES])[:, 1]))

LightGBM [0.323 1.    0.164] same as the classifier: True
XGBoost  [0.605 0.999 0.143] same as the classifier: True


From the same settings and seed, the boosters give the classifiers' probabilities, so the scikit-learn
classes are the native training with a `predict` that adds labels.

### Bands of cold

`pd.cut` sorts the lowest temperatures into bands, numbered 0 for below −10, 1 for −10 up to 0,
and 2 for 0 and above. Both classifiers see three classes in the labels and train one tree for each
class every round.

In [7]:
def bands(rows):
    return pd.cut(rows["tmin"], [-np.inf, -10, 0, np.inf], right=False, labels=False)

bands_train, bands_valid = bands(train), bands(valid)
print("days in each band:", np.bincount(bands_train), "training,", np.bincount(bands_valid), "validation")

lightgbm_bands = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_bands.fit(train[FEATURES], bands_train)
xgboost_bands = xgb.XGBClassifier(n_estimators=300, learning_rate=0.1, random_state=0)
xgboost_bands.fit(train[FEATURES], bands_train)
print("trees:", lightgbm_bands.booster_.num_trees(), "LightGBM,", len(xgboost_bands.get_booster().get_dump()), "XGBoost")

days in each band: [1601 6619 6780] training, [ 245 1074 1181] validation
trees: 900 LightGBM, 900 XGBoost


The coldest band has 1,601 training days, about a tenth of them, and each of the other two has more
than 6,600. Three classes and 300 rounds give 900 trees in both libraries, a tree for each band in
every round.

In [8]:
for name, model in (("LightGBM", lightgbm_bands), ("XGBoost", xgboost_bands)):
    probability = model.predict_proba(valid[FEATURES])
    label = model.predict(valid[FEATURES])
    print(f"{name:8} probabilities {probability.shape} | accuracy {accuracy_score(bands_valid, label):.3f} | log loss "
          f"{log_loss(bands_valid, probability):.3f} | AUC, each band against the rest "
          f"{roc_auc_score(bands_valid, probability, multi_class='ovr'):.3f}")
    print("  confusion matrix:", confusion_matrix(bands_valid, label).tolist())
print(lightgbm_bands.predict_proba(valid[FEATURES])[:3].round(3), lightgbm_bands.predict(valid[FEATURES])[:3])

LightGBM probabilities (2500, 3) | accuracy 0.911 | log loss 0.252 | AUC, each band against the rest 0.980
  confusion matrix: [[168, 77, 0], [20, 982, 72], [0, 54, 1127]]
XGBoost  probabilities (2500, 3) | accuracy 0.902 | log loss 0.249 | AUC, each band against the rest 0.979
  confusion matrix: [[165, 80, 0], [29, 967, 78], [0, 59, 1122]]
[[0.    0.516 0.484]
 [0.    1.    0.   ]
 [0.    0.176 0.824]] [1 1 2]


Each row of `predict_proba` now holds three probabilities that add up to 1, and `predict` returns the
band with the largest. `roc_auc_score` needs `multi_class='ovr'` for more than two classes, and then
averages the AUC of each band against the other two, 0.980 and 0.979. Accuracy is lower than for frost,
0.911 and 0.902, and the confusion matrices show that every mistake falls between neighboring bands,
and that no day below −10 is called a mild one, or the reverse. The first validation day
is split almost evenly between the middle band, 0.516, and the mild one, 0.484.

### Probabilities against how often frost happens

A probability of 0.7 should mean frost on about 70% of the days that get it. The cell sorts the
validation days into five groups by LightGBM's probability of frost and compares each group's mean
probability with its share of frost days.

In [9]:
probability = lightgbm_frost.predict_proba(valid[FEATURES])[:, 1]
groups = pd.cut(probability, [0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
check = pd.DataFrame({"probability": probability, "frost": frost_valid.to_numpy()}).groupby(groups, observed=False)
print(check.agg(days=("frost", "size"), mean_probability=("probability", "mean"), share_with_frost=("frost", "mean")).round(3))

               days  mean_probability  share_with_frost
(-0.001, 0.2]  1131             0.016             0.038
(0.2, 0.4]       48             0.287             0.479
(0.4, 0.6]       46             0.501             0.522
(0.6, 0.8]       53             0.714             0.717
(0.8, 1.0]     1222             0.989             0.975


Where LightGBM gives a probability between 0.4 and 0.8, the share of frost days is within 0.03 of the
mean probability. The two lowest groups have more frost than the model expected, 3.8% of days against
a mean probability of 1.6%, and 47.9% against 28.7%. Most days sit in the two outer groups, 1,131 and
1,222 of the 2,500, so the model is usually sure, and on these validation stations it underestimates
the chance of frost where it calls frost unlikely.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/10-classification-solutions.ipynb).

**1.** Turn LightGBM's probabilities of frost into labels at thresholds of 0.3 and 0.7 as well as 0.5, and print the accuracy on the validation stations at each.

In [10]:
# your code here

**2.** Train an `LGBMClassifier` for frost on `cloud_cover`, `wind_ms` and `day_of_year` alone, under a new name, and print its log loss and AUC on the validation stations beside the eight-feature model's.

In [11]:
# your code here

**3.** Add a band of cold below −20 to the bands, in a function of your own, leaving `bands` and `bands_train` as they are, train an `XGBClassifier` on them, and print the number of days in each band, the number of trees and the accuracy on the validation stations.

In [12]:
# your code here

**4.** For the first three validation days, print the raw scores `lightgbm_bands` gives each band with `raw_score=True`, and check that the softmax of the raw scores, e^score divided by the sum of e^score over the bands, gives `predict_proba`.

In [13]:
# your code here

## Common errors

### `ValueError: Classification metrics can't handle a mix of binary and continuous-multioutput targets`

In [14]:
accuracy_score(frost_valid, lightgbm_frost.predict_proba(valid[FEATURES]))

ValueError: Classification metrics can't handle a mix of binary and continuous-multioutput targets

`accuracy_score` compares labels, and `predict_proba` returns a table of probabilities, two columns for
each day, which scikit-learn calls continuous multioutput. Give accuracy the labels from `predict`, or
probabilities turned into labels at a threshold, and give log loss and AUC the probabilities.

In [15]:
print("accuracy:", round(accuracy_score(frost_valid, lightgbm_frost.predict(valid[FEATURES])), 3))

accuracy: 0.945


Given the labels from `predict`, `accuracy_score` reports 0.945, the accuracy the worked examples
printed.

### `LightGBMError: Number of classes should be specified and greater than 1 for multiclass training`

In [16]:
band_labels = pd.cut(train["tmin"], [-np.inf, -10, 0, np.inf], right=False, labels=False)
lgb.train({"objective": "multiclass", "verbose": -1}, lgb.Dataset(train[FEATURES], band_labels), num_boost_round=10)

[LightGBM] [Fatal] Number of classes should be specified and greater than 1 for multiclass training


LightGBMError: Number of classes should be specified and greater than 1 for multiclass training

LightGBM's native `"multiclass"` objective needs the number of classes as a setting, `num_class`, where
the scikit-learn class counts the classes in the labels. LightGBM prints the message from its own code
first, the `[LightGBM] [Fatal]` line, and Python then raises it. The next cell names the number of
classes in the settings, with `num_class`.

In [17]:
band_labels = pd.cut(train["tmin"], [-np.inf, -10, 0, np.inf], right=False, labels=False)
three_bands = lgb.train({"objective": "multiclass", "num_class": 3, "verbose": -1},
                        lgb.Dataset(train[FEATURES], band_labels), num_boost_round=10)
print(three_bands.num_trees(), "trees for 10 rounds")

30 trees for 10 rounds


With `num_class` set to 3, the booster trains three trees in every round, 30 for 10 rounds.

### No error, and an AUC from labels

In [18]:
label = lightgbm_frost.predict(valid[FEATURES])
probability = lightgbm_frost.predict_proba(valid[FEATURES])[:, 1]
print("AUC from labels:", round(roc_auc_score(frost_valid, label), 3), "| from probabilities:",
      round(roc_auc_score(frost_valid, probability), 3))

AUC from labels: 0.946 | from probabilities: 0.989


`roc_auc_score` accepts labels without complaint, and with only two values to order the days by, every
day labeled frost ties with every other, so the AUC falls from 0.989 to 0.946. AUC measures how well the
probabilities order the days, so give it `predict_proba`'s column for frost.

### No error, and probabilities below 0 and above 1

In [19]:
lightgbm_regression = lgb.train({"verbose": -1}, lgb.Dataset(train[FEATURES], frost_train), num_boost_round=300)
xgboost_regression = xgb.train({}, xgb.DMatrix(train[FEATURES], frost_train, enable_categorical=True), num_boost_round=300)
lightgbm_values = lightgbm_regression.predict(valid[FEATURES])
xgboost_values = xgboost_regression.predict(xgb.DMatrix(valid[FEATURES], enable_categorical=True))
print("LightGBM from", round(lightgbm_values.min(), 3), "to", round(lightgbm_values.max(), 3))
print("XGBoost from", round(float(xgboost_values.min()), 3), "to", round(float(xgboost_values.max()), 3))

LightGBM from -0.274 to 1.167
XGBoost from -0.438 to 1.442


Without an objective, both native interfaces train a regression on the labels, fitting squared error
to 0 and 1, and their predictions run from −0.274 to 1.167 and from −0.438 to 1.442, numbers no
probability can be. Name the objective, `"binary"` or `"binary:logistic"`, as the worked examples do.

In [20]:
lightgbm_binary = lgb.train({"objective": "binary", "verbose": -1}, lgb.Dataset(train[FEATURES], frost_train), num_boost_round=300)
xgboost_binary = xgb.train({"objective": "binary:logistic"}, xgb.DMatrix(train[FEATURES], frost_train, enable_categorical=True),
                           num_boost_round=300)
lightgbm_values = lightgbm_binary.predict(valid[FEATURES])
xgboost_values = xgboost_binary.predict(xgb.DMatrix(valid[FEATURES], enable_categorical=True))
print("LightGBM from", round(lightgbm_values.min(), 3), "to", round(lightgbm_values.max(), 3))
print("XGBoost from", round(float(xgboost_values.min()), 3), "to", round(float(xgboost_values.max()), 3))

LightGBM from 0.0 to 1.0
XGBoost from 0.0 to 1.0


With the objective named, the predictions of both models run from 0.0 to 1.0 at three places, inside
the range a probability can take.

## Recap

- You can train `LGBMClassifier` and `XGBClassifier` for two classes and for more, and read
  `predict_proba` as a row of probabilities for each day.
- You can score a classifier with accuracy from its labels, and log loss and AUC from its
  probabilities.
- You can turn the trees' raw scores into probabilities with the sigmoid, and train the same classifier in
  the native interfaces by naming its objective.
- You can check whether a model's probabilities match how often the class really happens.

## What is next

The **Imbalanced Classes** notebook asks about hard frost, below −15 degrees, which happens on fewer
than 2 days in 100, where accuracy says almost nothing about a model.

---

&#8592; **Previous:** [Categorical Features](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/09-categorical-features.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Imbalanced Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/11-imbalanced-classes.ipynb) &#8594;
